# Lesson 21: Building with Mistral Models

In this notebook, you will:
1. Build a **Cross-Language Korean-to-English FAISS (`IndexFlatL2`) RAG Pipeline** over Paul Graham's essay.
2. Run the local **Tekken tokenizer (`mistral-common[hf-hub]`)** with `get_current_weather` tool schemas.

In [ ]:
!pip install -q -U faiss-cpu sentence-transformers "mistral-common[hf-hub]" requests numpy

In [ ]:
import numpy as np, faiss
from sentence_transformers import SentenceTransformer

essay_chunks = [
    "Before college the two main things I worked on, outside of school, were writing and programming. I didn't write essays. I wrote what beginning writers were supposed to write then, and probably still are: short stories.",
    "The first programs I tried writing were on the IBM 1401 that our school district used for what was then called 'data processing.' This was in 9th grade, so I was 13 or 14.",
    "When I went to college I planned to study philosophy, but I found it boring and switched to AI and Lisp.",
]

embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
emb = np.array(embedder.encode(essay_chunks), dtype="float32")
index = faiss.IndexFlatL2(emb.shape[1])
index.add(emb)

korean_q = "저자가 대학에 오기 전에 주로 했던 두 가지 일은 무엇이었나요?"
q_emb = np.array(embedder.encode([korean_q]), dtype="float32")
D, I = index.search(q_emb, k=2)
print("Top retrieved English chunks for Korean query:")
for idx in I[0]:
    print(" -", essay_chunks[idx])

In [ ]:
from mistral_common.protocol.instruct.messages import UserMessage
from mistral_common.protocol.instruct.request import ChatCompletionRequest
from mistral_common.protocol.instruct.tool_calls import Function, Tool
from mistral_common.tokens.tokenizers.mistral import MistralTokenizer

tok = MistralTokenizer.from_hf_hub("mistralai/Mistral-Nemo-Instruct-2407")
req = ChatCompletionRequest(
    tools=[Tool(function=Function(name="get_current_weather", description="Get weather", parameters={"type": "object", "properties": {"location": {"type": "string"}}, "required": ["location"]}))],
    messages=[UserMessage(content="What's the weather like today in Paris")],
    model="mistralai/Mistral-Nemo-Instruct-2407",
)
print("Tekken token count:", len(tok.encode_chat_completion(req).tokens))